# 05 · 线性代数与 einsum Linear Algebra & einsum

> **能力标签**：SH3（NumPy 与向量化计算 / NumPy & vectorized computing）

## 目标 Objectives

完成本课后，你应该能够：

1. 用 `@` 运算符和 `np.dot` 做矩阵乘法，理解批量矩阵乘（batch matmul）。
2. 用 `np.linalg.solve` 解线性方程组，理解为何优于 `inv`。
3. 用 `np.linalg.norm` 计算向量/矩阵范数。
4. 用 `np.einsum` 表达 Gram 矩阵、批量点积等运算，并能读懂 einsum 字符串。
5. 推导并实现 OLS（普通最小二乘）估计量 $\hat{\beta} = (X^\top X)^{-1} X^\top y$。

> 对应能力：**SH3**


## 概念讲解 Concepts

### 矩阵乘法 Matrix Multiplication

```python
A = np.array([[1,2],[3,4]])   # shape (2,2)
B = np.array([[5,6],[7,8]])   # shape (2,2)
C = A @ B                     # = np.matmul(A, B)
```

- `@` 是 Python 3.5+ 的矩阵乘法运算符（`__matmul__`）。
- `np.dot` 对二维数组等价；对高维有不同语义，推荐始终用 `@`。

---

### np.linalg.solve vs inv

解 $Ax = b$：

```python
# 推荐：数值稳定，避免显式求逆
x = np.linalg.solve(A, b)

# 不推荐：求逆后相乘，数值误差大
x = np.linalg.inv(A) @ b
```

`solve` 使用 LU 分解，比先求逆再乘更稳定、更快。

---

### 范数 Norms

```python
np.linalg.norm(v)          # L2 范数（欧氏距离）
np.linalg.norm(v, ord=1)   # L1 范数
np.linalg.norm(A, 'fro')   # Frobenius 范数（矩阵）
```

---

### einsum 字符串语法

`np.einsum(subscripts, *operands)` 通过下标字符串描述运算：

| 表达式 | 含义 |
|--------|------|
| `"ij,jk->ik"` | 矩阵乘 $A B$ |
| `"ik,jk->ij"` | Gram 矩阵 $X X^\top$ |
| `"ii->"` | 矩阵的迹（trace）|
| `"ij->i"` | 行求和 |

规则：重复的下标会被**求和消去**（Einstein 求和约定）；箭头右边保留的下标即输出维度。

---

### OLS 估计量

普通最小二乘（Ordinary Least Squares）：

$$\hat{\beta} = (X^\top X)^{-1} X^\top y$$

避免显式求逆，用 `np.linalg.solve(X.T @ X, X.T @ y)` 实现。


## 示例 Worked Example 1：Gram 矩阵（gram）

In [ ]:
import numpy as np

# Gram 矩阵：G = X @ X.T，形状 (n, n)
# 每个 G[i,j] = X[i] · X[j]（行 i 与行 j 的内积）

def gram(X: np.ndarray) -> np.ndarray:
    """返回 Gram 矩阵 X @ X.T，用 einsum。"""
    X = np.asarray(X, dtype=float)
    return np.einsum("ik,jk->ij", X, X)
    # 等价于：X @ X.T

rng = np.random.default_rng(1)
X = rng.normal(size=(4, 6))   # 4 个样本，6 个特征

G = gram(X)
print("X.shape:", X.shape)
print("G = gram(X), shape:", G.shape)
print("G (前 4×4):\n", G.round(4))

# 验证：与 X @ X.T 相等
assert np.allclose(G, X @ X.T), "结果不一致！"
print("\n与 X @ X.T 一致：True")

# Gram 矩阵是半正定的（eigenvalues ≥ 0）
eigvals = np.linalg.eigvalsh(G)
print("特征值（应 ≥ 0）:", eigvals.round(6))


## 示例 Worked Example 2：OLS 估计量（ols_beta）

In [ ]:
import numpy as np

def ols_beta(X: np.ndarray, y: np.ndarray) -> np.ndarray:
    """普通最小二乘 β = (XᵀX)⁻¹Xᵀy，用 np.linalg.solve（不显式求逆）。"""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    return np.linalg.solve(X.T @ X, X.T @ y)

# 生成合成数据：y = 2*x1 + 3*x2 + 1 + noise
rng = np.random.default_rng(42)
n = 100
x1 = rng.normal(size=n)
x2 = rng.normal(size=n)
X = np.column_stack([np.ones(n), x1, x2])  # 含截距项的设计矩阵
true_beta = np.array([1.0, 2.0, 3.0])
y = X @ true_beta + rng.normal(scale=0.5, size=n)

beta_hat = ols_beta(X, y)
print("真实 β:    ", true_beta)
print("估计 β̂:   ", beta_hat.round(4))
print("误差:      ", (beta_hat - true_beta).round(4))

# 计算残差和 R²
y_hat = X @ beta_hat
residuals = y - y_hat
SS_res = residuals @ residuals
SS_tot = ((y - y.mean()) ** 2).sum()
R2 = 1 - SS_res / SS_tot
print(f"\nR² = {R2:.4f}  (应接近 1)")


In [ ]:
# einsum 的其他用法示例
import numpy as np

rng = np.random.default_rng(5)
A = rng.normal(size=(3, 4))
B = rng.normal(size=(4, 5))

# 矩阵乘法
C1 = np.einsum("ij,jk->ik", A, B)
C2 = A @ B
print("矩阵乘（einsum vs @）一致:", np.allclose(C1, C2))

# 矩阵的迹（trace）
M = rng.normal(size=(4, 4))
print("trace via einsum:", np.einsum("ii->", M))
print("trace via np.trace:", np.trace(M))

# 逐元素平方和（Frobenius norm²）
print("sum of squares:", np.einsum("ij,ij->", M, M))
print("via norm:", np.linalg.norm(M, 'fro')**2)

# 批量点积（batch dot product）
# shapes: (5, 3) 与 (5, 3) → 5 个三维向量的点积 → (5,)
u = rng.normal(size=(5, 3))
v = rng.normal(size=(5, 3))
dots = np.einsum("ij,ij->i", u, v)
print("\n批量点积 shape:", dots.shape)
print("批量点积（前 3）:", dots[:3].round(4))


## 动手 Exercise

用 `einsum` 实现**批量外积（batch outer product）**：
给定 shape `(B, m)` 的矩阵 $U$ 和 shape `(B, n)` 的矩阵 $V$，
返回每对向量的外积，输出 shape 为 `(B, m, n)`。


In [ ]:
import numpy as np

def batch_outer(U: np.ndarray, V: np.ndarray) -> np.ndarray:
    """批量外积：U[b] ⊗ V[b]，输出 shape (B, m, n)。"""
    return np.einsum("bi,bj->bij", U, V)

# 验证
rng = np.random.default_rng(9)
U = rng.normal(size=(3, 4))
V = rng.normal(size=(3, 5))
O = batch_outer(U, V)
print("O.shape:", O.shape)   # 应为 (3, 4, 5)

# 验证：手动计算第 0 个外积
O0_manual = np.outer(U[0], V[0])
print("与手动 outer 一致:", np.allclose(O[0], O0_manual))


## 延伸阅读 Further Reading

1. **NumPy linalg 文档**: <https://numpy.org/doc/stable/reference/routines.linalg.html>
2. **einsum 详解**: <https://numpy.org/doc/stable/reference/generated/numpy.einsum.html>
3. **《Linear Algebra Done Right》** — Sheldon Axler（理论基础）
4. **OLS 的矩阵推导**: 《统计学习方法》李航，第 3 章；或《ISLR》第 3 章
5. **einsum 可视化**: <https://ajcr.net/Basic-guide-to-einsum/>


## 关联练习 Related Assignments

本课对应两个练习包：

- **`w2-einsum`**（任务：`gram`）
- **`w2-ols`**（任务：`ols_beta`）

```bash
python check.py 03-einsum
python check.py 04-ols
```

> 能力标签：**SH3** · threshold ≥ 0.7
